# Steam Reviews 2021 — Data Mining Final Project
## Notebook 01 · Part 1, sections 1–3: Problem Definition, Dataset Description, Data Preprocessing

**Project type:** Text Mining — binary classification of review text (*recommended* vs *not recommended*).
**Dataset:** [Steam Reviews Dataset 2021 (Kaggle)](https://www.kaggle.com/datasets/najzeko/steam-reviews-2021)

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Problem definition, objectives, research questions |
| 2 | Dataset description (computed on the **full** 21.7M-row file) |
| 3 | Data preprocessing (sampling, types, duplicates, invalid values, text, missing values, outliers, transformations, split, encoding, scaling) |

**Before running:** put `steam_reviews.csv` into `data/raw/` (see `README.md`).
The first run reads the whole 8 GB file once in chunks (roughly 10–20 minutes, depending on the disk). The sample and the statistics are cached in `data/interim/`, so every later run takes seconds.

## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebook started from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from src import config as C
from src.preprocessing import (
    build_tabular_preprocessor,
    clean_text,
    convert_types,
    fix_invalid_values,
    iqr_outlier_table,
    load_or_scan,
    remove_duplicates,
    stratified_split,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")

for folder in (C.DATA_INTERIM, C.DATA_PROCESSED, C.MODELS_DIR, C.FIGURES_DIR, C.TABLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DECISIONS = []   # (step, what was done, why) -> exported at the end as the preprocessing log

def log_decision(step, action, reason):
    DECISIONS.append({"step": step, "action": action, "reason": reason})

def savefig(name):
    plt.tight_layout()
    plt.savefig(C.FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"Project root : {ROOT}")
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

## 1. Problem Definition

### 1.1 Problem and why it matters
Steam is the largest PC game store: popular games collect hundreds of thousands of user reviews. Every review carries a binary label chosen by its author — **Recommended 👍 / Not Recommended 👎** — plus free text.

* **Developers and publishers** need to find the negative feedback (bugs, performance, monetisation) fast; nobody can read 100,000 reviews by hand.
* **The same text appears without a label** on forums, Discord, Reddit, YouTube comments and support tickets. Steam reviews are a rare place where millions of texts are already labelled by their authors, so a model trained here can be transferred to unlabelled feedback.
* **Label–text mismatch detection:** reviews whose text contradicts the thumb (sarcasm, jokes, review bombing) are interesting both for the platform and for the error analysis of this project.

### 1.2 Objectives
1. Build a reproducible pipeline that turns the raw 8 GB file into a clean, representative and leakage-free modelling dataset.
2. Find out which properties of the text and of the reviewer distinguish negative reviews from positive ones (EDA, statistical tests).
3. Train and compare text classifiers (TF-IDF + at least three models) against a majority-class baseline, handling the strong class imbalance.
4. Interpret the final model (most indicative words) and analyse misclassified reviews.

### 1.3 Research questions and hypotheses
| # | Question / hypothesis | How it will be checked |
|---|---|---|
| **RQ1** | Can the review text alone predict the recommendation significantly better than a majority-class baseline? | F1-macro and PR-AUC of the *Not recommended* class, test set (Part 2) |
| **H1** | Players who do **not** recommend a game had **less playtime at the moment of the review** than players who recommend it. | Mann–Whitney U / ANOVA on `author_playtime_at_review` (EDA) |
| **H2** | Negative reviews are **longer** (more words) than positive ones. | Mann–Whitney U on review length (EDA) |
| **H3** | The recommendation rate is different for reviews written **during early access** and for **free copies**. | Chi-square test of independence (EDA) |

### 1.4 Why this dataset is appropriate
* **Real-world and large:** about 21.7 million reviews of 300+ games collected from the official Steam API.
* **Labelled by the authors themselves** — no manual annotation, no annotator bias.
* **Text + tabular data:** the text answers RQ1, the reviewer/playtime columns answer H1–H3.
* **Meets the course requirements** with a large margin (≥ 15 features, ≥ 40,000 records) even after sampling.
* **Realistic difficulties** that the course asks to handle: class imbalance, missing values, impossible values, duplicates, heavy-tailed distributions, many languages.

### 1.5 Project type
**Text Mining and NLP** (section 2 of Part 2 of the assignment): text cleaning → TF-IDF → cosine similarity → comparison of at least three classifiers → analysis of misclassified texts. Tabular features known at review time are added to the text features as a secondary input.

## 2. Dataset Description

### 2.1 Source
* **Name:** Steam Reviews Dataset 2021 — Kaggle, author *najzeko*
* **Link:** https://www.kaggle.com/datasets/najzeko/steam-reviews-2021
* **Collection:** Steam Web API (`appreviews` endpoint), January 2021
* **Licence:** GPL-2
* **File:** `steam_reviews.csv`, about 8 GB

The cell below reads the full file once (cached afterwards). All numbers in section 2 are calculated on the **full** dataset, not on the sample.

In [ ]:
raw_sample, full_stats, game_stats = load_or_scan()

In [ ]:
n_lang_reviews = sum(full_stats["target_selected_language"].values())
overview = pd.DataFrame(
    {
        "value": [
            f"{full_stats['total_rows']:,}",
            full_stats["n_columns_original"],
            len(full_stats["columns"]),
            full_stats["n_games"],
            len(full_stats["language_counts"]),
            f"{n_lang_reviews:,}",
            f"{full_stats['n_sampled']:,}",
            f"{full_stats['duplicate_review_ids']:,}",
            f"{full_stats['duplicate_author_game_pairs']:,}",
        ]
    },
    index=[
        "Records (reviews) in the full file",
        "Columns in the file",
        "Columns without the exported row index",
        "Games",
        "Review languages",
        f"{C.LANGUAGE.title()} reviews",
        f"Random sample ({C.SAMPLE_FRAC:.0%} of {C.LANGUAGE.title()} reviews)",
        "Repeated review_id in the full file",
        "Repeated (author, game) pairs in the full file",
    ],
)
overview

### 2.2 Features, their types and meaning

In [ ]:
data_dictionary = pd.DataFrame(
    [
        ("app_id", "identifier", "Steam ID of the game", "yes"),
        ("app_name", "categorical (nominal)", "Title of the game", "yes"),
        ("review_id", "identifier", "Unique ID of the review", "yes"),
        ("language", "categorical (nominal)", "Language of the review chosen by Steam", "yes"),
        ("review", "text", "Review text written by the player", "yes"),
        ("timestamp_created", "datetime (unix seconds)", "When the review was published", "yes"),
        ("timestamp_updated", "datetime (unix seconds)", "When the review was last edited", "no"),
        ("recommended", "boolean - TARGET", "Thumb up (True) or thumb down (False)", "target"),
        ("votes_helpful", "numeric (count)", "How many users marked the review as helpful", "no"),
        ("votes_funny", "numeric (count)", "How many users marked the review as funny", "no"),
        ("weighted_vote_score", "numeric (continuous 0-1)", "Steam helpfulness score", "no"),
        ("comment_count", "numeric (count)", "Number of comments under the review", "no"),
        ("steam_purchase", "boolean", "Game was bought on Steam (not a key)", "yes"),
        ("received_for_free", "boolean", "Author got the game for free", "yes"),
        ("written_during_early_access", "boolean", "Review written while the game was in early access", "yes"),
        ("author_steamid", "identifier", "Steam ID of the author", "yes"),
        ("author_num_games_owned", "numeric (count)", "Games in the author's library", "approximately"),
        ("author_num_reviews", "numeric (count)", "Reviews written by the author", "approximately"),
        ("author_playtime_forever", "numeric (minutes)", "Total playtime when the data was collected", "no"),
        ("author_playtime_last_two_weeks", "numeric (minutes)", "Playtime in the 2 weeks before collection", "no"),
        ("author_playtime_at_review", "numeric (minutes)", "Playtime at the moment of writing the review", "yes"),
        ("author_last_played", "datetime (unix seconds)", "Last time the author played the game", "no"),
    ],
    columns=["column", "type", "meaning", "known when the review is written?"],
).set_index("column")

assert set(data_dictionary.index) == set(full_stats["columns"]), "unexpected column set"
print("Dots in the original names were replaced by underscores (author.num_reviews -> author_num_reviews).")
print("The extra first column of the CSV ('Unnamed: 0') is a row number exported by pandas and is dropped.\n")
print(data_dictionary["type"].str.split(" ").str[0].value_counts().rename("number of columns").to_string())
data_dictionary

The last column is important for the whole project. Votes, comments, later playtime and edits appear **after** the review is published. A model that is supposed to classify a freshly written review must not use them — otherwise it would "see the future" (look-ahead leakage). They are kept for EDA only (section 3.9).

### 2.3 Missing values in the full dataset

In [ ]:
missing_full = (
    pd.Series(full_stats["missing_per_column"], name="missing")
    .to_frame()
    .assign(**{"missing_%": lambda d: (100 * d["missing"] / full_stats["total_rows"]).round(3)})
    .sort_values("missing", ascending=False)
)
display(missing_full)

with_missing = missing_full[missing_full["missing"] > 0]
if with_missing.empty:
    print("No missing values in the full file.")
else:
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(with_missing) + 1.2))
    sns.barplot(x=with_missing["missing_%"], y=with_missing.index, color="#c0392b", ax=ax)
    ax.set_xlabel("% of rows missing (full dataset)")
    ax.set_ylabel("")
    ax.set_title("Missing values per column — full dataset")
    savefig("01_missing_values_full")

Missing values are concentrated in a few columns (the review text and the playtime / last-played fields). Note that "missing" here only means an empty cell in the CSV; *impossible* values (overflow numbers, future dates) are additional hidden missing values that are found in section 3.4.

### 2.4 Languages

In [ ]:
lang = pd.Series(full_stats["language_counts"], name="reviews")
lang_share = (100 * lang / lang.sum()).round(2).rename("share_%")
display(pd.concat([lang, lang_share], axis=1).head(12))

fig, ax = plt.subplots(figsize=(8, 4.5))
top = lang_share.head(12)
sns.barplot(x=top.values, y=top.index, color="#2c7fb8", ax=ax)
ax.set_xlabel("% of all reviews")
ax.set_ylabel("")
ax.set_title("Top-12 review languages — full dataset")
savefig("02_languages_full")

English is by far the largest language. The project uses **English reviews only**: one language means one tokenizer, one stop-word list and comparable TF-IDF vocabularies. Mixing Chinese, Russian and English words in one bag-of-words model would create a vocabulary in which a word's weight depends on the language mix rather than on sentiment.

### 2.5 Target variable and class distribution

In [ ]:
def class_table(counts: dict, name: str) -> pd.DataFrame:
    s = pd.Series(counts, name=name)
    return pd.DataFrame({name: s, f"{name} %": (100 * s / s.sum()).round(2)})

target_full = pd.concat(
    [class_table(full_stats["target_all_languages"], "all languages"),
     class_table(full_stats["target_selected_language"], C.LANGUAGE)],
    axis=1,
)
display(target_full)

t = full_stats["target_selected_language"]
imbalance_ratio = t["recommended"] / max(t["not_recommended"], 1)
print(f"Imbalance ({C.LANGUAGE}): {imbalance_ratio:.1f} positive reviews per 1 negative review")
print(f"A model that always answers 'recommended' would have accuracy "
      f"{t['recommended'] / (t['recommended'] + t['not_recommended']):.1%} — and be useless.")

plot_df = (
    target_full[["all languages %", f"{C.LANGUAGE} %"]]
    .drop(index="missing", errors="ignore")
    .rename(columns=lambda c: c.replace(" %", ""))
    .T
)
ax = plot_df.plot(kind="bar", stacked=True, color=["#27ae60", "#c0392b"], figsize=(6, 4), rot=0)
ax.set_ylabel("% of reviews")
ax.set_title("Class distribution of the target 'recommended'")
ax.legend(title="", loc="lower right")
savefig("03_class_distribution_full")

**Target:** `recommended` (1 = recommended, 0 = not recommended). The classes are strongly imbalanced, so in Part 2:
* accuracy is **not** used as the main metric — F1-macro, recall and PR-AUC of the minority class are;
* the split is stratified, and imbalance is handled with class weights (and compared with resampling).

### 2.6 Aggregation by game

In [ ]:
games = game_stats.copy()
games["negative_%_all"] = (100 * (1 - games["pos_all"] / games["n_all"])).round(2)
lang_ok = games["n_lang"] > 0
games.loc[lang_ok, f"negative_%_{C.LANGUAGE}"] = (
    100 * (1 - games.loc[lang_ok, "pos_lang"] / games.loc[lang_ok, "n_lang"])
).round(2)
games = games.sort_values("n_lang", ascending=False)
display(games.head(15))

top15 = games.head(15).sort_values(f"negative_%_{C.LANGUAGE}")
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.barplot(x=top15[f"negative_%_{C.LANGUAGE}"], y=top15.index, color="#c0392b", ax=ax)
ax.set_xlabel(f"% negative {C.LANGUAGE} reviews")
ax.set_ylabel("")
ax.set_title(f"Share of negative reviews — 15 games with most {C.LANGUAGE} reviews")
savefig("04_negative_share_by_game")

print(f"Reviews per game ({C.LANGUAGE}): median {games['n_lang'].median():,.0f}, "
      f"max {games['n_lang'].max():,.0f}, min {games['n_lang'].min():,.0f}")

Games differ a lot both in the number of reviews and in the share of negative ones. Two consequences:
1. the sample must keep each game's share — guaranteed by simple random sampling and checked in 3.1;
2. `app_name` is a useful feature (a review of a game with many negative reviews is more likely to be negative), but its encoding must be learned on the training part only (3.11).

In [ ]:
print("Key facts for the report:")
print(f"* {full_stats['total_rows']:,} reviews, {len(full_stats['columns'])} columns, "
      f"{full_stats['n_games']} games, {len(full_stats['language_counts'])} languages")
print(f"* {C.LANGUAGE.title()}: {n_lang_reviews:,} reviews "
      f"({100 * n_lang_reviews / full_stats['total_rows']:.1f}% of all)")
print(f"* Positive share ({C.LANGUAGE}): "
      f"{100 * t['recommended'] / (t['recommended'] + t['not_recommended']):.1f}%")
cols_missing = ", ".join(with_missing.index) if not with_missing.empty else "none"
print(f"* Columns with empty cells: {cols_missing}")
print(f"* Repeated review_id: {full_stats['duplicate_review_ids']:,}; "
      f"repeated (author, game): {full_stats['duplicate_author_game_pairs']:,}")

## 3. Data Preprocessing

Every decision is written into a log (`DECISIONS`) and exported to `reports/tables/preprocessing_log.csv` — the table goes straight into the report.

**Order of steps and the leakage rule.** Steps 3.1–3.9 work row by row with fixed rules (they do not learn anything from the data), so they can be applied before the split. Everything that *learns* from the data — quantiles for clipping, medians for imputation, means/standard deviations for scaling, the list of games for one-hot encoding — is fitted **on the training set only** (3.11).

### 3.1 Sampling
**Why sample at all?** 21.7 million rows (≈ 8 GB of text) do not fit into the RAM of a normal laptop, and TF-IDF + SVM on ~10 million English texts would take hours per experiment. Hyper-parameter tuning needs dozens of experiments.

**How:** simple random sampling — during the single pass over the file every English review is kept with probability `SAMPLE_FRAC = 0.02`, using a fixed seed (`random_state = 42`), so the result is reproducible. Simple random sampling preserves the class balance and the share of every game in expectation; the table below verifies this. We deliberately **do not balance** the classes here: the test set must reflect the real distribution; imbalance is handled inside the model (Part 2).

In [ ]:
sample_rec = pd.to_numeric(
    raw_sample["recommended"].astype("string").str.lower().map({"true": 1, "false": 0}),
    errors="coerce",
)
full_pos_share = t["recommended"] / (t["recommended"] + t["not_recommended"])
sample_pos_share = sample_rec.mean()

game_share_full = games["n_lang"] / games["n_lang"].sum()
game_share_sample = raw_sample["app_name"].value_counts(normalize=True)
aligned = pd.concat([game_share_full.rename("full"), game_share_sample.rename("sample")], axis=1).fillna(0)

representativeness = pd.DataFrame(
    {
        f"full {C.LANGUAGE} data": [f"{n_lang_reviews:,}", f"{full_pos_share:.2%}", "-", "-"],
        "sample": [
            f"{len(raw_sample):,}",
            f"{sample_pos_share:.2%}",
            f"{aligned['full'].corr(aligned['sample']):.4f}",
            f"{100 * (aligned['full'] - aligned['sample']).abs().max():.3f} p.p.",
        ],
    },
    index=["rows", "share of 'recommended'",
           "correlation of per-game shares", "largest per-game share difference"],
)
display(representativeness)
log_decision("3.1 Sampling",
             f"Kept English reviews only; simple random sample of {C.SAMPLE_FRAC:.0%} "
             f"(seed {C.RANDOM_STATE}) -> {len(raw_sample):,} rows",
             "8 GB does not fit in memory; one language = one consistent vocabulary; "
             "random sampling keeps class and game proportions (verified)")

### 3.2 Data types and technical columns
All columns were read as text (the only safe way to read a 8 GB file with occasional broken cells). Now they get proper types:
* booleans `'True'/'False'` → 1/0;
* counts, playtime, timestamps → numbers; anything that cannot be parsed becomes `NaN` and is counted;
* `author_steamid` stays **text**: a 17-digit number does not fit into `float64` exactly, so two different players could get the same ID after conversion.

In [ ]:
df = convert_types(raw_sample)

numeric_cols = C.COUNT_COLS + C.TIMESTAMP_COLS + ["weighted_vote_score", "review_id", "app_id"] + C.BOOL_COLS
parse_failures = pd.Series(
    {c: int((raw_sample[c].notna() & df[c].isna()).sum()) for c in numeric_cols},
    name="values that could not be parsed -> NaN",
)
display(parse_failures[parse_failures > 0].to_frame() if parse_failures.sum() else "No unparseable values.")

assert df["language"].nunique() == 1, "sample must contain one language only"
df = df.drop(columns=["language"])

display(df.dtypes.rename("dtype").to_frame().T)
log_decision("3.2 Types",
             "Converted booleans to 0/1, numbers/timestamps to numeric, kept Steam IDs as text; "
             "dropped 'Unnamed: 0' (exported row index) and 'language' (constant after filtering)",
             "Correct types are needed for statistics and models; a constant or row-number column carries no information")

### 3.3 Duplicate removal
Three kinds of duplicates:
1. **exact copies** of a row;
2. the same `review_id` twice (the same review collected twice, e.g. before and after an edit);
3. the same **author + game** pair twice — Steam allows only one review per game per account, so a repeat is an older copy. The most recent version is kept.

In [ ]:
n_before = len(df)
df, dup_report = remove_duplicates(df)
display(dup_report)
print(f"Rows: {n_before:,} -> {len(df):,} (removed {n_before - len(df):,})")
print(f"For reference, the full file contains {full_stats['duplicate_review_ids']:,} repeated review_id and "
      f"{full_stats['duplicate_author_game_pairs']:,} repeated (author, game) pairs; "
      "most of them fall outside a 2% sample, the ones inside are removed above.")
log_decision("3.3 Duplicates",
             f"Removed exact copies, repeated review_id and repeated (author, game) pairs: "
             f"{n_before - len(df):,} rows",
             "Duplicates over-weight some reviews and can end up in both train and test (optimistic evaluation)")

### 3.4 Invalid (impossible) values
Some numbers in this dataset are not extreme but **impossible** — they are technical markers or collection errors. They are hidden missing values and are turned into `NaN` (or repaired by a rule), *before* the outlier analysis, otherwise they would destroy every mean and standard deviation.

| Rule | Reason |
|---|---|
| any count ≥ 2³¹ (e.g. 4,294,967,295 = 2³²−1, 4,398,046,511,104 = 2⁴²) | overflow / "unknown" marker of the Steam API |
| `author_num_games_owned` or `author_num_reviews` > 50,000 | Steam had about 50,000 games in 2021 |
| `author_playtime_last_two_weeks` > 20,160 min | 14 days contain only 20,160 minutes |
| `weighted_vote_score` outside [0, 1] | the score is defined on [0, 1] |
| playtime at review > total playtime | total playtime is measured later, cannot be smaller → capped |
| dates before Steam's launch (2003) or after the data collection (Jan 2021) | impossible dates; a wrong edit date is replaced by the creation date |
| `author_last_played = 0` | Steam's code for "never / unknown" |

In [ ]:
check_cols = C.COUNT_COLS + ["weighted_vote_score", "timestamp_updated", "author_last_played"]
max_before = df[check_cols].max()
df, invalid_report = fix_invalid_values(df)
max_after = df[check_cols].max()

display(invalid_report[invalid_report["values_fixed"] > 0].reset_index(drop=True))
display(pd.DataFrame({"max before": max_before, "max after": max_after}).style.format("{:,.0f}"))
log_decision("3.4 Invalid values",
             f"{int(invalid_report['values_fixed'].sum()):,} impossible values set to NaN or repaired by domain rules",
             "Overflow markers and impossible dates are errors, not real observations; "
             "they would distort statistics, scaling and models")

### 3.5 Review text: cleaning, empty and broken reviews
The text is the main input of the model, so it gets its own treatment.

**Cleaning (label-independent, applied to every row the same way):** lower-case → remove URLs → remove Steam formatting tags (`[h1]`, `[b]`, `[spoiler]`, …) → collapse whitespace. The original text is kept in `review` for error analysis; the clean one goes to `review_clean`. Tokenisation, stop words and n-grams are decided later together with TF-IDF, because they are model hyper-parameters.

**Rows removed:**
* empty text (`NaN`) — a missing text cannot be imputed: inventing words would invent sentiment;
* longer than 8,000 characters — Steam does not accept longer reviews, so these are glued/broken rows;
* no Latin letters after cleaning (`"."`, `"👍"`, digits only) — TF-IDF would turn them into an all-zero vector, the model could only guess the majority class.

In [ ]:
n0 = len(df)
text_missing = df["review"].isna()
too_long = df["review"].fillna("").str.len() > C.STEAM_REVIEW_MAX_CHARS
df["review_clean"] = clean_text(df["review"])
no_letters = ~df["review_clean"].str.contains(r"[a-z]", regex=True)

text_report = pd.DataFrame(
    {"rows": [int(text_missing.sum()), int((too_long & ~text_missing).sum()),
              int((no_letters & ~text_missing & ~too_long).sum())]},
    index=["empty text (NaN)", f"longer than {C.STEAM_REVIEW_MAX_CHARS:,} characters",
           "no Latin letters after cleaning"],
)
display(text_report)

changed = df.loc[~text_missing & (df["review"].fillna("").str.lower().str.strip() != df["review_clean"]),
                 ["review", "review_clean"]]
print(f"Reviews changed by cleaning: {len(changed):,} ({100 * len(changed) / n0:.1f}%). Examples:")
display(changed.head(5))

df = df.loc[~(text_missing | too_long | no_letters)].copy()
print(f"Rows: {n0:,} -> {len(df):,}")

dup_texts = df["review_clean"].duplicated(keep=False)
print(f"\nIdentical texts written by different players: {dup_texts.mean():.1%} of rows. Most common:")
display(df["review_clean"].value_counts().head(8).rename("count").to_frame())
log_decision("3.5 Text",
             "Lower-case, removed URLs and BBCode tags, collapsed spaces -> review_clean; "
             f"dropped {n0 - len(df):,} reviews that were empty, broken (> 8,000 chars) or had no letters",
             "Formatting and links carry no sentiment; empty/broken texts cannot be represented by TF-IDF")
log_decision("3.5 Text",
             "Kept identical short texts written by different players ('good game', ...)",
             "They are different real reviews with their own labels; removing them would distort the class "
             "distribution of short reviews")

### 3.6 Missing values
Strategy per column group:

| Columns | Treatment | Why |
|---|---|---|
| `recommended` (target) | drop row | a row without the answer cannot be used for training or evaluation |
| `author_playtime_at_review` | fill with `author_playtime_forever` of the same row, if available | playtime at the review ≤ total playtime; for most reviews (written recently) they are equal or close — a row-level rule, nothing learned from other rows |
| `app_name` | fill with `"unknown"` | a category of its own; nothing learned from data |
| remaining NaN in numeric model features | **median** imputation + a binary "was missing" flag, fitted on train inside the pipeline (3.11) | median is robust to heavy tails; the flag keeps the information that the value was missing |
| remaining NaN in boolean model features | most frequent value, fitted on train | |
| post-review columns (votes, later playtime, last played) | left as NaN | used only in EDA, where pandas/statistics skip NaN; not used by the model |

In [ ]:
def missing_table(frame):
    m = frame.isna().sum()
    return pd.DataFrame({"missing": m, "missing_%": (100 * m / len(frame)).round(3)})[m > 0]

before = missing_table(df)
display(before.rename(columns=lambda c: f"{c} (before)"))

n0 = len(df)
df = df.loc[df[C.TARGET].notna()].copy()
dropped_target = n0 - len(df)

fill_mask = df["author_playtime_at_review"].isna() & df["author_playtime_forever"].notna()
df.loc[fill_mask, "author_playtime_at_review"] = df.loc[fill_mask, "author_playtime_forever"]
df["app_name"] = df["app_name"].fillna("unknown")
df[C.TARGET] = df[C.TARGET].astype("int8")

after = missing_table(df)
print(f"Rows dropped because the target was missing: {dropped_target:,}")
print(f"author_playtime_at_review filled from total playtime: {int(fill_mask.sum()):,}")
display(after.rename(columns=lambda c: f"{c} (after)") if len(after) else "No missing values left.")

left_in_model = [c for c in C.NUM_FEATURES + C.BOOL_FEATURES if df[c].isna().any()]
print(f"Model features that still have NaN -> imputed on train inside the pipeline: {left_in_model or 'none'}")
log_decision("3.6 Missing values",
             f"Dropped {dropped_target:,} rows without target; filled {int(fill_mask.sum()):,} "
             "playtime_at_review values from total playtime; app_name NaN -> 'unknown'; "
             "other model features: median / most-frequent imputation + missing flag, fitted on train",
             "Row-level rules use only the same row (no leakage); learned imputation is fitted on train only; "
             "median is robust to heavy tails")

### 3.7 Outlier analysis
After 3.4 every remaining value is *possible*. The question now is whether extreme values are errors or real behaviour. Tukey's rule (outside Q1 − 1.5·IQR … Q3 + 1.5·IQR) is calculated on the raw scale and on the log scale.

In [ ]:
outliers = iqr_outlier_table(df, C.COUNT_COLS)
display(outliers)
outliers.to_csv(C.TABLES_DIR / "outlier_analysis.csv")

plot_cols = C.NUM_FEATURES + ["author_playtime_forever"]
fig, axes = plt.subplots(2, len(plot_cols), figsize=(4 * len(plot_cols), 6.5))
for j, col in enumerate(plot_cols):
    x = df[col].dropna()
    sns.boxplot(x=x, ax=axes[0, j], color="#e67e22", fliersize=1)
    axes[0, j].set_title(f"{col}\nraw", fontsize=9)
    axes[0, j].set_xlabel("")
    sns.boxplot(x=np.log1p(x), ax=axes[1, j], color="#16a085", fliersize=1)
    axes[1, j].set_title("log1p", fontsize=9)
    axes[1, j].set_xlabel("")
fig.suptitle("Outliers before and after the log transformation", y=1.02)
savefig("05_outliers_raw_vs_log")

**Reading the table:**
* Counts and playtime are **heavy-tailed** (skewness far above 1): a few players have thousands of games or tens of thousands of hours. On the raw scale Tukey's rule flags a large share of rows as outliers, which is not plausible for "errors" — it is simply the shape of the distribution.
* After `log1p` the share of outliers drops sharply and the skewness is close to 0 — the tail becomes ordinary.
* Columns such as `votes_helpful` / `comment_count` are 0 for most reviews, so IQR = 0 and every non-zero value looks like an outlier. That is a property of the data, not an error.

**Decision:** real extreme values are **not deleted** — a 10,000-hour player is a real customer and their opinion matters; deleting them would bias the data towards casual players. Instead:
1. impossible values were already removed (3.4);
2. model features are transformed with `log1p` (3.8);
3. they are winsorised at the 0.1 % / 99.9 % quantiles **learned on the training set** (3.11), which limits the influence of the last few points on linear models and SVM.

In [ ]:
log_decision("3.7 Outliers",
             "IQR analysis on raw and log scale; extreme but possible values kept; "
             "model features: log1p + clipping to the train 0.1%/99.9% quantiles",
             "Heavy tails are real player behaviour, not errors; log + clipping reduce their influence "
             "without throwing away customers")

### 3.8 Variable transformations
* **Log transformation** `log1p(x) = log(1 + x)` for counts and playtime: handles zeros, turns multiplicative differences (10 h vs 100 h vs 1,000 h) into equal steps and makes the distributions close to symmetric. It is applied inside the model pipeline (3.11) so that exactly the same transformation is applied to new data.
* **Timestamps** → `datetime` columns, needed for time-based EDA (review date, edit delay).
* **Booleans** → 0/1 integers, target → `int8`.

In [ ]:
for col, new in [("timestamp_created", "created_at"),
                 ("timestamp_updated", "updated_at"),
                 ("author_last_played", "last_played_at")]:
    df[new] = pd.to_datetime(df[col], unit="s", errors="coerce")
df = df.drop(columns=C.TIMESTAMP_COLS)

skew = pd.DataFrame({
    "skew raw": df[C.NUM_FEATURES].skew(),
    "skew after log1p": np.log1p(df[C.NUM_FEATURES]).skew(),
}).round(2)
display(skew)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
x = df["author_playtime_at_review"].dropna()
sns.histplot(x / 60, bins=60, ax=axes[0], color="#e67e22")
axes[0].set_title("Playtime at review (hours) — raw")
axes[0].set_xlabel("hours")
sns.histplot(np.log1p(x), bins=60, ax=axes[1], color="#16a085")
axes[1].set_title("Playtime at review — log1p(minutes)")
axes[1].set_xlabel("log1p(minutes)")
savefig("06_log_transformation")

print(f"Review dates: {df['created_at'].min():%Y-%m-%d} ... {df['created_at'].max():%Y-%m-%d}")
log_decision("3.8 Transformations",
             "log1p for counts/playtime (in pipeline); unix timestamps -> datetime; booleans -> 0/1",
             "log1p removes the strong right skew (see table); datetimes are needed for time-based EDA")

### 3.9 Feature roles and leakage control
The model is meant to classify a review **at the moment it is written**. Therefore only information that exists at that moment may be used as input. Columns created later are kept in the dataset for EDA, but they are excluded from the model input.

In [ ]:
roles = {C.TARGET: "target", "review_clean": "model input: text (TF-IDF in Part 2)",
         "review": "original text (for error analysis only)"}
roles.update({c: "model input: numeric" for c in C.NUM_FEATURES})
roles.update({c: "model input: boolean" for c in C.BOOL_FEATURES})
roles.update({c: "model input: categorical" for c in C.CAT_FEATURES})
roles.update({c: "identifier (not a feature)" for c in C.ID_COLS})
roles["created_at"] = "date (EDA / feature engineering)"
post = ["votes_helpful", "votes_funny", "weighted_vote_score", "comment_count",
        "author_playtime_forever", "author_playtime_last_two_weeks", "last_played_at", "updated_at"]
roles.update({c: "after publication -> EDA only (leakage)" for c in post})

missing_roles = set(df.columns) - set(roles)
assert not missing_roles, f"columns without a role: {missing_roles}"
role_table = pd.Series(roles, name="role").reindex(df.columns).to_frame()
display(role_table.sort_values("role"))
log_decision("3.9 Leakage",
             "Model input = text + 3 numeric + 3 boolean + game; votes, comments, later playtime and "
             "edit dates are EDA-only",
             "These values appear after the review is published; using them would let the model see the future")

### 3.10 Train / validation / test split
Stratified **70 / 15 / 15** split with a fixed seed. Stratification keeps exactly the same class ratio in every part, which matters with a strong imbalance.
* **train** — fitting the preprocessing and the models;
* **validation** — hyper-parameter tuning and model selection (together with cross-validation on train);
* **test** — touched **once**, for the final evaluation of the chosen model.

The split is done *before* any learned transformation, so nothing from validation/test leaks into the preprocessing.

In [ ]:
train_df, val_df, test_df = stratified_split(df)
split_table = pd.DataFrame({
    "rows": [len(train_df), len(val_df), len(test_df)],
    "share_%": [100 * len(p) / len(df) for p in (train_df, val_df, test_df)],
    "recommended_%": [100 * p[C.TARGET].mean() for p in (train_df, val_df, test_df)],
}, index=["train", "validation", "test"]).round(2)
display(split_table)

assert not set(train_df["review_id"]) & set(test_df["review_id"]), "review in both train and test!"
assert not set(train_df["review_id"]) & set(val_df["review_id"]), "review in both train and validation!"
log_decision("3.10 Split",
             f"Stratified 70/15/15 split (seed {C.RANDOM_STATE}): "
             f"{len(train_df):,} / {len(val_df):,} / {len(test_df):,} rows",
             "Same class ratio in every part; test set is reserved for the single final evaluation")

### 3.11 Categorical encoding and scaling
One `ColumnTransformer` (defined in `src/preprocessing.py`, reused in Part 2):

| Group | Steps |
|---|---|
| numeric (`author_num_games_owned`, `author_num_reviews`, `author_playtime_at_review`) | clip to train 0.1 %/99.9 % quantiles → `log1p` → median imputation → **StandardScaler** (mean 0, std 1) |
| missing flags | one 0/1 column per numeric feature: 1 = the value was missing and was imputed |
| boolean (`steam_purchase`, `received_for_free`, `written_during_early_access`) | most-frequent imputation; already 0/1, no scaling needed |
| categorical (`app_name`) | **One-Hot encoding**; games with < 20 training reviews are merged into one *infrequent* column; unseen games → that column |

**Why One-Hot and not Label/Ordinal encoding:** games have no order — numbering them 1, 2, 3 would tell a linear model that game 3 is "three times" game 1. **Why not target encoding:** it uses the label and leaks it unless it is done with careful cross-fitting; with ~300 games One-Hot is small and sparse anyway.
**Why scaling:** logistic regression with regularisation, SVM and KNN depend on the scale of features; without it a column measured in minutes dominates a 0/1 column. TF-IDF features are already on a 0–1 scale.

The transformer is **fitted on train only** and then applied to validation and test.

In [ ]:
preprocessor = build_tabular_preprocessor()
X_train_tab = preprocessor.fit_transform(train_df)
X_val_tab = preprocessor.transform(val_df)
X_test_tab = preprocessor.transform(test_df)

feature_names = preprocessor.get_feature_names_out()
print(f"Tabular feature matrix: train {X_train_tab.shape}, validation {X_val_tab.shape}, test {X_test_tab.shape}")
print(f"Output columns: {len(feature_names)}  "
      f"(numeric: {sum(n.startswith('num__') for n in feature_names)}, "
      f"missing flags: {sum(n.startswith('missing__') for n in feature_names)}, "
      f"boolean: {sum(n.startswith('bool__') for n in feature_names)}, "
      f"one-hot games: {sum(n.startswith('cat__') for n in feature_names)})")

num_idx = [i for i, n in enumerate(feature_names) if n.startswith("num__")]
def dense(m):
    return m.toarray() if hasattr(m, "toarray") else np.asarray(m)
scaling_check = pd.DataFrame({
    "train mean": dense(X_train_tab[:, num_idx]).mean(axis=0),
    "train std": dense(X_train_tab[:, num_idx]).std(axis=0),
    "validation mean": dense(X_val_tab[:, num_idx]).mean(axis=0),
    "validation std": dense(X_val_tab[:, num_idx]).std(axis=0),
}, index=feature_names[num_idx]).round(3)
display(scaling_check)

ohe = preprocessor.named_transformers_["cat"]
infrequent = ohe.infrequent_categories_[0]
n_infrequent = 0 if infrequent is None else len(infrequent)
print(f"Games with their own column: {len(ohe.categories_[0]) - n_infrequent}; "
      f"merged into 'infrequent': {n_infrequent}")

joblib.dump(preprocessor, C.PREPROCESSOR_PATH)
print(f"Saved fitted preprocessor -> {C.PREPROCESSOR_PATH.relative_to(ROOT)}")
log_decision("3.11 Encoding & scaling",
             "One-Hot for app_name (rare games -> 'infrequent'); StandardScaler for log-transformed "
             "numeric features; everything fitted on train only",
             "Games are nominal (no order); distance/regularisation-based models need comparable scales; "
             "fitting on train only prevents leakage")

Train means are 0 and standard deviations 1 by construction; validation values are close to them, which shows that the training statistics generalise (no distribution shift between the parts).

### 3.12 Saving the processed data

In [ ]:
outputs = {"train": train_df, "val": val_df, "test": test_df}
for name, part in outputs.items():
    path = C.DATA_PROCESSED / f"{name}.parquet"
    part.to_parquet(path, index=False)
    print(f"{name:<5} {len(part):>9,} rows -> {path.relative_to(ROOT)}  ({path.stat().st_size / 1e6:,.1f} MB)")

### 3.13 Preprocessing log and final dataset summary

In [ ]:
decision_log = pd.DataFrame(DECISIONS)
decision_log.to_csv(C.TABLES_DIR / "preprocessing_log.csv", index=False)
display(decision_log.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

funnel = pd.Series({
    "full dataset": full_stats["total_rows"],
    f"{C.LANGUAGE} reviews": n_lang_reviews,
    f"random {C.SAMPLE_FRAC:.0%} sample": full_stats["n_sampled"],
    "after cleaning (modelling dataset)": len(df),
    "train": len(train_df), "validation": len(val_df), "test": len(test_df),
}, name="rows").map("{:,}".format).to_frame()
display(funnel)
print(f"Final modelling dataset: {len(df):,} reviews, {df.shape[1]} columns, "
      f"positive share {100 * df[C.TARGET].mean():.1f}%")

**Result of sections 1–3:** a clean, representative, leakage-safe English sample split into train / validation / test, plus a fitted tabular preprocessor.

**Next steps (separate notebooks):**
* Part 1 §4 — EDA on the training set: summary statistics, distributions, correlations, ≥ 5 visualisations, statistical tests for H1–H3.
* Part 1 §5 — feature engineering (review length, `playtime_at_review / playtime_forever`, exclamation marks, …), feature selection, PCA / TruncatedSVD.
* Part 2 — TF-IDF on `review_clean`, baseline, ≥ 3 models, tuning, evaluation, error analysis.